In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id=dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01.enviroment-config

In [0]:
%run ../00-common/03.silver_helpers

In [0]:

bronze_table=f"{catalog_name}.{bronze_schema}.sprints"
silver_table=f"{catalog_name}.{silver_schema}.sprints"

In [0]:
from pyspark.sql import functions as F

In [0]:
sprints_df=(
    (spark.table(bronze_table)).filter(F.col("batch_id")==v_batch_id)
    .select("season",
                      "round","constructorId","driverId","date","raceName","grid","laps","number","points","position","positionText","status","ingestion_timestamp","source_file","batch_id")
    .withColumnsRenamed({
        "constructorId":"constructor_id",
        "driverId":"driver_id",
        "raceName":"race_name",
        "date":"race_date",
        "grid":"grid_position",
        "laps": "completed_laps",
        "number":"car_number",
        "position":"final_position",
        "positionText":"final_position_text"}))
     

In [0]:
#for season,round,constructor_id,driver_id

sprints_valid_df=(sprints_df
                  .filter(F.col("season").isNotNull() & 
                          F.col("driver_id").isNotNull() &  
                          F.col("constructor_id").isNotNull() & 
                          F.col("round").isNotNull() 
                  )
                  .dropDuplicates(["season","round","constructor_id","driver_id"])
                  )

In [0]:
display(sprints_df.count()-sprints_valid_df.count())


In [0]:
#trasform values of race name to title case
sprints_final_df=sprints_valid_df.withColumn("race_name",F.initcap(F.col("race_name")))

In [0]:
write_to_silver(
    input_df=sprints_final_df,
    target_table=silver_table,
    merge_condition="track_id = s.track_id AND driver_id = s.driver_id AND race_year = s.race_year AND race_number = s.race_number",
    columns_to_update=["race_name","race_date","grid_position","completed_laps","number","points","final_position","final_position_text","ingestion_timestamp","source_file","batch_id"] 
)

In [0]:
# (
#     sprints_final_df
#     .write
#     .format("delta")
#     .mode("overwrite")
#     .saveAsTable(silver_table)
# )

In [0]:
display(spark.table(silver_table))